# Migration Fear Index - Exploratory Data Analysis

This notebook explores the Migration Fear Index dataset covering multiple countries.

**Dataset:** `migration-fear.xlsx`

**Description:** Quarterly migration-related EPU and fear indices for UK, Germany, USA, and France from 1990 onwards.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_PATH = Path('../../datasets/raw/migration-fear.xlsx')
CSV_OUTPUT_PATH = Path('../../datasets/raw/migration-fear.csv')

## 1. Load and Clean Data

In [ ]:
# Load Excel file
df = pd.read_excel(DATA_PATH)

print("Raw data (first 10 rows):")
display(df.head(10))

# Clean column names
df.columns = df.columns.str.strip()

# Create date column from year and quarter
df['date'] = pd.to_datetime(df['year'].astype(str) + '-' + ((df['quarter'] - 1) * 3 + 1).astype(str) + '-01')

# Sort by date
df = df.sort_values('date').reset_index(drop=True)

print("\nCleaned dataset:")
display(df.head())
print("\nData types:")
print(df.dtypes)
print("\nColumn names:")
print(df.columns.tolist())

## 2. Save to CSV

In [ ]:
# Save cleaned data to CSV
df.to_csv(CSV_OUTPUT_PATH, index=False)
print(f"✓ Data saved to: {CSV_OUTPUT_PATH}")
print(f"  Rows saved: {len(df)}")

## 3. Data Overview

In [ ]:
print("Dataset Shape:", df.shape)
print("\nDate range:", df['date'].min(), "to", df['date'].max())
print("Total quarters:", len(df))
print("\nDataset Info:")
df.info()

## 4. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
print("Missing Values Summary:")
display(missing_df[missing_df['Missing Count'] > 0])

## 5. Descriptive Statistics

In [ ]:
# Get all index columns (excluding year, quarter, date)
index_cols = [col for col in df.columns if col not in ['year', 'quarter', 'date']]

print("Descriptive Statistics:")
display(df[index_cols].describe())

## 6. Time Series - EPU Migrant Indices

In [ ]:
# Plot EPU migrant indices for all countries
epu_cols = [col for col in df.columns if 'epu_migrant' in col]

fig, ax = plt.subplots(figsize=(16, 6))
for col in epu_cols:
    country = col.split('_')[0].upper()
    ax.plot(df['date'], df[col], linewidth=1.5, label=country, alpha=0.8)

ax.set_title('Migration-Related EPU Indices by Country', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('EPU Migrant Index', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Time Series - Fear Indices

In [ ]:
# Plot fear indices for all countries
fear_cols = [col for col in df.columns if 'fear' in col]

fig, ax = plt.subplots(figsize=(16, 6))
for col in fear_cols:
    country = col.split('_')[0].upper()
    ax.plot(df['date'], df[col], linewidth=1.5, label=country, alpha=0.8)

ax.set_title('Migration Fear Indices by Country', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Fear Index', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Country-by-Country Analysis

In [ ]:
# Create subplots for each country
countries = ['uk', 'germany', 'usa', 'france']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, country in enumerate(countries):
    epu_col = f'{country}_epu_migrant_index'
    fear_col = f'{country}_fear_index'
    
    ax = axes[idx]
    ax2 = ax.twinx()
    
    # Plot EPU on left axis
    line1 = ax.plot(df['date'], df[epu_col], color='blue', linewidth=1.5, label='EPU Migrant', alpha=0.7)
    ax.set_ylabel('EPU Migrant Index', color='blue', fontsize=10)
    ax.tick_params(axis='y', labelcolor='blue')
    
    # Plot Fear on right axis
    line2 = ax2.plot(df['date'], df[fear_col], color='red', linewidth=1.5, label='Fear', alpha=0.7)
    ax2.set_ylabel('Fear Index', color='red', fontsize=10)
    ax2.tick_params(axis='y', labelcolor='red')
    
    ax.set_title(f'{country.upper()} - Migration Indices', fontsize=12, fontweight='bold')
    ax.set_xlabel('Date', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Combine legends
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax.legend(lines, labels, loc='upper left')

plt.tight_layout()
plt.show()

## 9. Correlation Analysis

In [ ]:
# Correlation matrix
corr_matrix = df[index_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('Correlation Matrix - Migration Indices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Distribution Comparison

In [ ]:
# Box plots for EPU indices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# EPU Migrant Indices
epu_data = [df[col].dropna() for col in epu_cols]
epu_labels = [col.split('_')[0].upper() for col in epu_cols]
axes[0].boxplot(epu_data, labels=epu_labels)
axes[0].set_title('Distribution of EPU Migrant Indices', fontsize=12, fontweight='bold')
axes[0].set_ylabel('EPU Migrant Index')
axes[0].grid(True, alpha=0.3)

# Fear Indices
fear_data = [df[col].dropna() for col in fear_cols]
fear_labels = [col.split('_')[0].upper() for col in fear_cols]
axes[1].boxplot(fear_data, labels=fear_labels)
axes[1].set_title('Distribution of Fear Indices', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Fear Index')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Summary Statistics by Country

In [ ]:
# Summary statistics for each country
for country in countries:
    epu_col = f'{country}_epu_migrant_index'
    fear_col = f'{country}_fear_index'
    
    print(f"\n{'='*60}")
    print(f"{country.upper()} - Summary Statistics")
    print(f"{'='*60}")
    
    print(f"\nEPU Migrant Index:")
    print(f"  Mean: {df[epu_col].mean():.2f}")
    print(f"  Median: {df[epu_col].median():.2f}")
    print(f"  Std Dev: {df[epu_col].std():.2f}")
    print(f"  Min: {df[epu_col].min():.2f}")
    print(f"  Max: {df[epu_col].max():.2f}")
    
    print(f"\nFear Index:")
    print(f"  Mean: {df[fear_col].mean():.2f}")
    print(f"  Median: {df[fear_col].median():.2f}")
    print(f"  Std Dev: {df[fear_col].std():.2f}")
    print(f"  Min: {df[fear_col].min():.2f}")
    print(f"  Max: {df[fear_col].max():.2f}")
    
    # Correlation between EPU and Fear for this country
    corr = df[[epu_col, fear_col]].corr().iloc[0, 1]
    print(f"\nCorrelation (EPU vs Fear): {corr:.3f}")

## 12. Summary

In [ ]:
print("=" * 60)
print("KEY FINDINGS - MIGRATION FEAR INDEX")
print("=" * 60)
print(f"\n1. Dataset Coverage:")
print(f"   - Start: {df['date'].min().strftime('%Y-Q%q')}")
print(f"   - End: {df['date'].max().strftime('%Y-Q%q')}")
print(f"   - Total Quarters: {len(df)}")
print(f"   - Countries: UK, Germany, USA, France")

print(f"\n2. Indices Tracked:")
print(f"   - EPU Migrant Index (4 countries)")
print(f"   - Fear Index (4 countries)")
print(f"   - Total: {len(index_cols)} indices")

print(f"\n3. Data Quality:")
print(f"   - Total Missing Values: {df[index_cols].isnull().sum().sum()}")
print(f"   - CSV file saved: {CSV_OUTPUT_PATH}")

print(f"\n4. Cross-Country Insights:")
# Find country with highest average EPU
avg_epu = {country: df[f'{country}_epu_migrant_index'].mean() for country in countries}
max_epu_country = max(avg_epu, key=avg_epu.get)
print(f"   - Highest avg EPU: {max_epu_country.upper()} ({avg_epu[max_epu_country]:.2f})")

# Find country with highest average Fear
avg_fear = {country: df[f'{country}_fear_index'].mean() for country in countries}
max_fear_country = max(avg_fear, key=avg_fear.get)
print(f"   - Highest avg Fear: {max_fear_country.upper()} ({avg_fear[max_fear_country]:.2f})")

print("\n" + "=" * 60)